# 09. Shortcut learning versus plot context

This notebook compares review, metadata, and plot feature blocks, then measures how much performance changes when explicit spoiler-warning phrases are removed. Matched conditions use the same rows, grouping strategy, folds, model settings, and metrics.

The main model is logistic regression because it is already implemented, works with sparse TF-IDF features, and permits coefficient inspection.


## 1. Setup

`SAMPLE_SIZE` limits only model-selection rows; final fitting uses the complete outer training partition. Both review-text-grouped and movie-disjoint settings are included below.


In [1]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start Jupyter from the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import clean_data
from src.logistic_regression import create_logistic_regression
from src.metrics import evaluate
from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import create_train_test_split
from src.text_features import create_feature_transformer, generate_tfidf_folds, oversample_positive_indices

RANDOM_STATE = 67
N_FOLDS = 5
SAMPLE_SIZE = 10_000
OVERSAMPLE_POSITIVE = True
MAX_ITER = 1_000
TOL = 1e-3
TFIDF_PARAMS = {"ngram_range": (1, 1), "max_features": 5_000}
MODEL_GRID = [{"penalty": "l2", "C": 0.1}, {"penalty": "l2", "C": 1.0}]
ALL_FEATURES = INITIAL_FEATURES.copy()
FEATURE_SETS = {
    "review_only": ["review_text"],
    "review_and_summary": ["review_text", "review_summary"],
    "plot_only": ["plot_summary", "plot_synopsis"],
    "metadata_only": ["rating", "genre", "duration", "days_since_release"],
    "review_plus_metadata": ["review_text", "review_summary", "rating", "genre", "duration", "days_since_release"],
    "review_plus_plot": ["review_text", "review_summary", "plot_summary", "plot_synopsis"],
    "all_features": ALL_FEATURES,
}
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shortcut_plot_comparison"


## 2. Load and prepare both split settings

The movie metadata is joined through `movie_id`. Labels and grouping IDs are retained for evaluation and grouping but are not model features.


In [2]:
clean_data()
_, standard_train_reviews, standard_test_reviews = create_train_test_split("review_text")
_, movie_train_reviews, movie_test_reviews = create_train_test_split("movie_id")
movies = pd.read_json(PROJECT_ROOT / "data" / "cleaned" / "IMDB_movie_details.json", lines=True, convert_dates=False)

split_reviews = {
    "review_text_grouped": (standard_train_reviews, standard_test_reviews, "review_text"),
    "movie_disjoint": (movie_train_reviews, movie_test_reviews, "movie_id"),
}
prepared = {}
for name, (train_reviews, test_reviews, group_by) in split_reviews.items():
    train = prepare_features(train_reviews, movies)
    test = prepare_features(test_reviews, movies)
    train["is_spoiler"] = train_reviews["is_spoiler"].to_numpy()
    test["is_spoiler"] = test_reviews["is_spoiler"].to_numpy()
    train["movie_id"] = train_reviews["movie_id"].to_numpy()
    test["movie_id"] = test_reviews["movie_id"].to_numpy()
    prepared[name] = (train, test, group_by)
print({name: (len(train), len(test), group_by) for name, (train, test, group_by) in prepared.items()})


Loading raw data...
Cleaning movie IDs...
Cleaning complete.
Cleaned reviews saved to: C:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\data\cleaned\IMDB_reviews.json
Cleaned movie details saved to: C:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\data\cleaned\IMDB_movie_details.json
Train: 459,130 (80.00%), spoiler rate 26.30%
Test:  114,783 (20.00%), spoiler rate 26.30%
Train: 459,361 (80.04%), spoiler rate 26.31%
Test:  114,552 (19.96%), spoiler rate 26.26%


C:\Users\quanh\AppData\Local\Temp\ipykernel_16156\933304610.py:4: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  movies = pd.read_json(PROJECT_ROOT / "data" / "cleaned" / "IMDB_movie_details.json", lines=True, convert_dates=False)


{'review_text_grouped': (459130, 114783, 'review_text'), 'movie_disjoint': (459361, 114552, 'movie_id')}


## 3. Run matched feature and cue-removal experiments

Fold-local encoders are created through `generate_tfidf_folds`. For review-containing configurations, both original and cue-removed conditions are run. Plot-only and metadata-only configurations run once because spoiler-cue removal does not affect their inputs.


In [3]:
def select_rows(data, size):
    if size is None or size >= len(data):
        return data.reset_index(drop=True)
    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.sort(rng.choice(len(data), size=size, replace=False))
    return data.iloc[indices].reset_index(drop=True)

def block_columns(feature_slices, names):
    blocks = [np.arange(feature_slices[name].start, feature_slices[name].stop) for name in names
              if feature_slices[name].start != feature_slices[name].stop]
    return np.concatenate(blocks).astype(int) if blocks else np.array([], dtype=int)

def run_condition(train_data, test_data, group_by, feature_name, feature_names, remove_spoilers):
    tuning_data = select_rows(train_data, SAMPLE_SIZE)
    candidates = []
    for params in MODEL_GRID:
        scores_by_fold = []
        for fold in generate_tfidf_folds(
            tuning_data, feature_columns=ALL_FEATURES, group_by=group_by, k=N_FOLDS,
            random_state=RANDOM_STATE, remove_spoilers=remove_spoilers,
            oversample_positive=OVERSAMPLE_POSITIVE, **TFIDF_PARAMS,
        ):
            columns = block_columns(fold["feature_slices"], feature_names)
            if not len(columns):
                raise ValueError(f"No usable encoded columns for {feature_names}.")
            model = create_logistic_regression(**params, random_state=RANDOM_STATE, max_iter=MAX_ITER, tol=TOL)
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                model.fit(fold["X_train"][:, columns], fold["y_train"])
            scores = evaluate(fold["y_val"], model.predict(fold["X_val"][:, columns]), model.predict_proba(fold["X_val"][:, columns])[:, 1])
            scores_by_fold.append(scores)
        candidates.append({
            "params": params,
            "cv_macro_f1_mean": np.mean([score["macro_f1"] for score in scores_by_fold]),
            "cv_macro_f1_std": np.std([score["macro_f1"] for score in scores_by_fold], ddof=1),
        })
    best = max(candidates, key=lambda row: row["cv_macro_f1_mean"])

    transformer = create_feature_transformer(train_data, feature_names, remove_spoilers=remove_spoilers, **TFIDF_PARAMS)
    X_train = csr_matrix(transformer.fit_transform(train_data))
    X_test = csr_matrix(transformer.transform(test_data))
    y_train = train_data["is_spoiler"].to_numpy()
    if OVERSAMPLE_POSITIVE:
        indices = oversample_positive_indices(y_train, RANDOM_STATE)
        X_train, y_train = X_train[indices], y_train[indices]
    model = create_logistic_regression(**best["params"], random_state=RANDOM_STATE, max_iter=MAX_ITER, tol=TOL)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    metrics = evaluate(test_data["is_spoiler"], predictions, probabilities)
    metrics["average_precision"] = metrics.pop("pr_auc")
    metrics.update({"feature_set": feature_name, "remove_spoilers": remove_spoilers,
                    "best_C": best["params"]["C"], "cv_macro_f1_mean": best["cv_macro_f1_mean"],
                    "cv_macro_f1_std": best["cv_macro_f1_std"]})
    return metrics, model, transformer

results = []
fitted_models = {}
for split_name, (train_data, test_data, group_by) in prepared.items():
    for feature_name, feature_names in FEATURE_SETS.items():
        conditions = [False, True] if {"review_text", "review_summary"} & set(feature_names) else [False]
        for remove_spoilers in conditions:
            metrics, model, transformer = run_condition(train_data, test_data, group_by, feature_name, feature_names, remove_spoilers)
            metrics.update({"split": split_name, "group_by": group_by, "train_rows": len(train_data),
                            "test_rows": len(test_data), "test_movies": test_data["movie_id"].nunique()})
            results.append(metrics)
            fitted_models[(split_name, feature_name, remove_spoilers)] = (model, transformer)
            print(split_name, feature_name, remove_spoilers, f"macro-F1={metrics['macro_f1']:.4f}")
results_df = pd.DataFrame(results)
display(results_df[["split", "feature_set", "remove_spoilers", "accuracy", "precision", "recall", "f1", "macro_f1", "average_precision", "best_C", "cv_macro_f1_mean"]])


review_text_grouped review_only False macro-F1=0.6621
review_text_grouped review_only True macro-F1=0.6538
review_text_grouped review_and_summary False macro-F1=0.6618
review_text_grouped review_and_summary True macro-F1=0.6508
review_text_grouped plot_only False macro-F1=0.5531
review_text_grouped metadata_only False macro-F1=0.5271
review_text_grouped review_plus_metadata False macro-F1=0.6722
review_text_grouped review_plus_metadata True macro-F1=0.6551
review_text_grouped review_plus_plot False macro-F1=0.6673
review_text_grouped review_plus_plot True macro-F1=0.6543
review_text_grouped all_features False macro-F1=0.6778
review_text_grouped all_features True macro-F1=0.6637
movie_disjoint review_only False macro-F1=0.6701
movie_disjoint review_only True macro-F1=0.6539
movie_disjoint review_and_summary False macro-F1=0.6709
movie_disjoint review_and_summary True macro-F1=0.6550
movie_disjoint plot_only False macro-F1=0.5244
movie_disjoint metadata_only False macro-F1=0.5419
movie_d

,split,feature_set,remove_spoilers,accuracy,precision,recall,f1,macro_f1,average_precision,best_C,cv_macro_f1_mean
0,review_text_grouped,review_only,False,0.702987,0.456261,0.675104,0.544517,0.662084,0.558672,1.0,0.637147
1,review_text_grouped,review_only,True,0.694258,0.445899,0.670200,0.535511,0.653823,0.537131,1.0,0.629475
2,review_text_grouped,review_and_summary,False,0.702787,0.455967,0.674110,0.543984,0.661769,0.558946,0.1,0.625418
3,review_text_grouped,review_and_summary,True,0.691278,0.442328,0.667119,0.531951,0.650816,0.535045,0.1,0.622561
4,review_text_grouped,plot_only,False,0.594914,0.335206,0.549611,0.416431,0.553111,0.348566,0.1,0.535755
5,review_text_grouped,metadata_only,False,0.566730,0.309769,0.527249,0.390255,0.527119,0.316685,0.1,0.516810
6,review_text_grouped,review_plus_metadata,False,0.714879,0.470572,0.673315,0.553976,0.672222,0.578219,1.0,0.627122
7,review_text_grouped,review_plus_metadata,True,0.695530,0.447418,0.671327,0.536965,0.655081,0.540579,1.0,0.615721
8,review_text_grouped,review_plus_plot,False,0.706716,0.461428,0.689382,0.552828,0.667317,0.571312,0.1,0.624483
9,review_text_grouped,review_plus_plot,True,0.693448,0.445597,0.678648,0.537968,0.654300,0.545165,0.1,0.622193


## 4. Quantify plot gains and shortcut dependence

`plot_gain` compares review plus plot against review-only. `cue_drop` compares the original condition against its cue-removed counterpart. These are descriptive differences between controlled experiments.


In [4]:
def row_for(split, feature_set, remove_spoilers):
    rows = results_df[(results_df["split"] == split) & (results_df["feature_set"] == feature_set) & (results_df["remove_spoilers"] == remove_spoilers)]
    if len(rows) != 1:
        raise ValueError(f"Expected one result row, found {len(rows)}")
    return rows.iloc[0]

effects = []
for split in results_df["split"].unique():
    review = row_for(split, "review_only", False)
    review_plot = row_for(split, "review_plus_plot", False)
    effects.append({
        "split": split,
        "plot_gain_macro_f1": review_plot["macro_f1"] - review["macro_f1"],
        "plot_gain_average_precision": review_plot["average_precision"] - review["average_precision"],
        "review_cue_drop_macro_f1": review["macro_f1"] - row_for(split, "review_only", True)["macro_f1"],
        "review_plot_cue_drop_macro_f1": review_plot["macro_f1"] - row_for(split, "review_plus_plot", True)["macro_f1"],
    })
effects_df = pd.DataFrame(effects)
display(effects_df)
display(results_df.sort_values(["split", "macro_f1"], ascending=[True, False])[["split", "feature_set", "remove_spoilers", "macro_f1", "f1", "recall", "average_precision"]])


,split,plot_gain_macro_f1,plot_gain_average_precision,review_cue_drop_macro_f1,review_plot_cue_drop_macro_f1
0,review_text_grouped,0.005233,0.012640,0.008261,0.013016
1,movie_disjoint,0.003718,-0.000454,0.016152,0.018822


,split,feature_set,remove_spoilers,macro_f1,f1,recall,average_precision
18,movie_disjoint,review_plus_metadata,False,0.675934,0.549683,0.638939,0.571652
20,movie_disjoint,review_plus_plot,False,0.673774,0.546000,0.632489,0.561515
14,movie_disjoint,review_and_summary,False,0.670939,0.546742,0.648181,0.562868
22,movie_disjoint,all_features,False,0.670194,0.537520,0.611809,0.537567
12,movie_disjoint,review_only,False,0.670056,0.545230,0.645455,0.561969
19,movie_disjoint,review_plus_metadata,True,0.660361,0.531410,0.627735,0.535602
15,movie_disjoint,review_and_summary,True,0.655032,0.529608,0.641831,0.530091
21,movie_disjoint,review_plus_plot,True,0.654952,0.526251,0.628333,0.523316
13,movie_disjoint,review_only,True,0.653904,0.529907,0.647550,0.531150
23,movie_disjoint,all_features,True,0.653560,0.519221,0.605326,0.495307


## 5. Inspect coefficients and save results

For the all-feature model, inspect the strongest coefficients before and after cue removal. This helps identify explicit warning terms and movie-/plot-specific vocabulary.


In [6]:
def get_feature_names_from_transformer(transformer):
    """Extract names from a fitted ColumnTransformer with nested pipelines."""
    feature_names = []

    for name, fitted_transformer, columns in transformer.transformers_:
        if isinstance(fitted_transformer, str):
            if fitted_transformer == "drop":
                continue
            branch_names = [str(column) for column in columns]
        elif hasattr(fitted_transformer, "named_steps"):
            final_transformer = fitted_transformer.steps[-1][1]
            if hasattr(final_transformer, "get_feature_names_out"):
                branch_names = final_transformer.get_feature_names_out(columns)
            else:
                branch_names = [str(column) for column in columns]
        else:
            branch_names = fitted_transformer.get_feature_names_out(columns)

        feature_names.extend(
            f"{name}__{branch_name}" for branch_name in branch_names
        )

    return np.asarray(feature_names, dtype=object)


for key, (model, transformer) in fitted_models.items():
    if key[1] != "all_features":
        continue
    coefficients = model.coef_.ravel()
    names = get_feature_names_from_transformer(transformer)
    if len(names) != len(coefficients):
        raise ValueError(
            f"Feature-name count ({len(names)}) does not match coefficient count "
            f"({len(coefficients)}) for {key}."
        )
    order = np.argsort(coefficients)
    print(f"{key[0]}, cues removed={key[2]}: strongest negative coefficients")
    display(pd.DataFrame({"feature": names[order[:15]], "coefficient": coefficients[order[:15]]}))
    print(f"{key[0]}, cues removed={key[2]}: strongest positive coefficients")
    display(pd.DataFrame({"feature": names[order[-15:][::-1]], "coefficient": coefficients[order[-15:][::-1]]}))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_DIR / "feature_ablation_metrics.csv", index=False)
effects_df.to_csv(OUTPUT_DIR / "shortcut_plot_effects.csv", index=False)
print(f"Saved results to {OUTPUT_DIR}")


review_text_grouped, cues removed=False: strongest negative coefficients


,feature,coefficient
0,review_summary__vidiot,-1.760413
1,review_summary__nutshell,-1.686298
2,review_text__script,-1.637604
3,plot_summary__edu,-1.629460
4,review_text__role,-1.576136
5,review_text__movies,-1.437677
6,review_text__director,-1.415021
7,review_text__performance,-1.381680
8,review_text__direction,-1.316727
9,review_text__action,-1.303294


review_text_grouped, cues removed=False: strongest positive coefficients


,feature,coefficient
0,review_text__spoilers,10.907381
1,review_text__spoiler,10.778171
2,review_summary__spoilers,5.790761
3,review_summary__spoiler,4.376210
4,review_text__scene,3.800865
5,review_text__killed,3.625918
6,review_text__ending,3.569535
7,days_since_release__days_since_release,3.420313
8,review_text__end,3.401654
9,review_text__death,3.207639


review_text_grouped, cues removed=True: strongest negative coefficients


,feature,coefficient
0,review_summary__vidiot,-1.850944
1,review_summary__nutshell,-1.820802
2,plot_summary__edu,-1.648432
3,review_text__script,-1.622566
4,review_text__role,-1.621884
5,review_text__director,-1.498220
6,review_text__movies,-1.468614
7,review_text__performance,-1.392819
8,review_text__direction,-1.350345
9,review_text__action,-1.332140


... (7 outputs left out)

,feature,coefficient
0,review_summary__vidiot,-2.318604
1,review_summary__nutshell,-2.254419
2,plot_summary__edu,-1.933205
3,review_text__role,-1.716204
4,review_text__director,-1.643639
5,review_text__performance,-1.629085
6,review_text__script,-1.474686
7,review_text__com,-1.444839
8,review_text__theater,-1.438023
9,review_text__year,-1.397902


movie_disjoint, cues removed=True: strongest positive coefficients


,feature,coefficient
0,days_since_release__days_since_release,4.418231
1,review_text__killed,4.227357
2,review_text__ending,3.833079
3,review_text__end,3.761920
4,review_text__scene,3.646782
5,review_text__dies,3.588670
6,review_text__death,3.297270
7,review_text__warning,3.035400
8,review_text__kill,3.019091
9,review_text__died,2.770738


Saved results to c:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\outputs\shortcut_plot_comparison


## 6. Discussions

### Main findings

The experiments show that review text is the strongest individual feature source for spoiler classification. Plot-only features achieve macro-F1 of 0.5531 for the review-text-grouped split and 0.5244 for the movie-disjoint split, while metadata-only features achieve 0.5271 and 0.5419 respectively. Combining review text with metadata or plot information gives only modest gains over review text alone. The best review-text-grouped result is obtained with all features (macro-F1 0.6778), whereas the best movie-disjoint result is obtained with review text plus metadata (macro-F1 0.6759).

The small plot gains are important: adding plot features to review features increases macro-F1 by only 0.0052 in the review-text-grouped setting and 0.0037 in the movie-disjoint setting. The corresponding average-precision gain is 0.0126 for the review-text-grouped split and is effectively zero (-0.0005) for the movie-disjoint split. This suggests that plot context provides limited additional predictive information beyond the review text under these settings.

### Evidence of shortcut learning

Removing explicit spoiler-warning cues decreases performance in every review-containing configuration. For the review-only model, macro-F1 falls by 0.0083 in the review-text-grouped split and by 0.0162 in the movie-disjoint split. For review plus plot, the drop is 0.0130 and 0.0188 respectively. The larger drops under movie-disjoint evaluation indicate that warning phrases are particularly useful when the model must generalise to unseen movies.

The coefficient inspection supports this interpretation. Terms such as `spoiler`, `spoilers`, `warning`, `killed`, `ending`, `death`, and `scene` receive strong positive coefficients. These features are plausible spoiler indicators, but they also show that the classifier can rely on explicit lexical cues rather than understanding plot events.

### Split-strategy takeaway

The review-text-grouped split has nearly balanced spoiler rates and is useful for measuring performance when duplicate review text is kept together. The movie-disjoint split is the more demanding evaluation for generalisation to previously unseen movies. Its lower performance for several feature configurations, together with the larger cue-removal penalty, suggests that results from the conventional split may overstate robustness to new movies.

### Limitations and future work

These findings are based on logistic regression with TF-IDF features, a 10,000-row model-selection sample, a small hyperparameter grid, and one fixed split seed. The results should therefore be interpreted as controlled comparisons rather than a final benchmark. Future work should repeat the experiments across multiple movie-disjoint splits, report uncertainty intervals, test stronger semantic models, and evaluate whether plot features become more useful after removing explicit spoiler cues or using richer plot representations.
